# NSCLC Classifier Training & Evaluation — v2

v2 changes from v1:
- CV AUROC (mean ± std) now printed alongside test AUROC in all comparison cells
- DCA uses logistic regression (radiomics-only) as the best model, consistent with the primary analysis

**Inputs:**
- `integrated_features_lasso_12m.csv` — 49 features (45 radiomics + 4 clinical)
- `integrated_features_lasso_6m.csv` — 185 features (178 radiomics + 7 clinical)

**Models compared:** Logistic Regression, XGBoost, Random Forest

**Stream ablation:** All features vs Radiomics-only vs Clinical-only

**Censoring:** Patients censored before the landmark are excluded (unknown label)

**Metrics:** AUROC (primary), CV AUROC, Brier score, calibration slope, DCA

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd()))

from radiomics_pipeline import (
    ResponsePredictor, ModelComparison,
    ClassifierConfig, ModelType, CensoringStrategy,
)

HERE = Path.cwd()
RESULTS = {}
print('Setup complete')

## 1. Load Data

In [ ]:
df_12m = pd.read_csv(HERE / 'integrated_features_lasso_12m.csv')
df_6m  = pd.read_csv(HERE / 'integrated_features_lasso_6m.csv')

print(f'12-month dataset: {len(df_12m)} patients, {len(df_12m.columns)-4} features')
print(f'6-month dataset : {len(df_6m)} patients, {len(df_6m.columns)-4} features')

# Survival arrays (shared)
time  = df_12m['survival_days'].values
event = df_12m['event_occurred'].values

# Feature columns by stream
def get_feature_cols(df, prefix=None):
    exclude = {'patient_id','survival_days','event_occurred','landmark_12m','landmark_6m'}
    cols = [c for c in df.columns if c not in exclude]
    if prefix:
        cols = [c for c in cols if c.startswith(prefix)]
    return cols

# 12-month streams
feat_12m_all  = get_feature_cols(df_12m)
feat_12m_rad  = get_feature_cols(df_12m, 'rad_')
feat_12m_clin = get_feature_cols(df_12m, 'clin_')

# 6-month streams  
feat_6m_all   = get_feature_cols(df_6m)
feat_6m_rad   = get_feature_cols(df_6m, 'rad_')
feat_6m_clin  = get_feature_cols(df_6m, 'clin_')

print(f'\n12-month features: {len(feat_12m_all)} total '
      f'({len(feat_12m_rad)} radiomics, {len(feat_12m_clin)} clinical)')
print(f'6-month features : {len(feat_6m_all)} total '
      f'({len(feat_6m_rad)} radiomics, {len(feat_6m_clin)} clinical)')

## 2. Censoring Check

Patients censored *before* the landmark have unknown label and are excluded from classification.

In [ ]:
for lm_days, label in [(365, '12-month'), (180, '6-month')]:
    died_before   = ((event == 1) & (time < lm_days)).sum()
    died_after    = ((event == 1) & (time >= lm_days)).sum()
    cens_before   = ((event == 0) & (time < lm_days)).sum()  # EXCLUDED
    cens_after    = ((event == 0) & (time >= lm_days)).sum()
    usable        = len(time) - cens_before
    print(f'{label} landmark ({lm_days}d):')
    print(f'  Died before landmark (label 0) : {died_before}')
    print(f'  Died after landmark  (label 1) : {died_after}')
    print(f'  Censored after landmark(label 1): {cens_after}')
    print(f'  Censored before landmark (EXCL): {cens_before}')
    print(f'  Usable for classification       : {usable}')
    print()

## 3. Model Comparison — 12-Month OS Landmark (Primary)

Compares four model types across three feature sets:
- **All**: radiomics + clinical (23 features)
- **Radiomics only**: 18 features
- **Clinical baseline**: 5 features (age, gender, histology, T stage, N stage)

In [ ]:
def run_comparison(df, feat_cols, time, event, landmark_days,
                   label='', n_folds=5, seed=42):
    """Run model comparison and return results table."""
    X = df[feat_cols].values
    names = feat_cols

    comparison = ModelComparison(
        landmark_days=landmark_days,
        model_types=[
            ModelType.LOGISTIC,
            ModelType.XGBOOST,
            ModelType.RANDOM_FOREST,
            ModelType.COX,
        ],
        n_folds=n_folds,
        random_state=seed,
    )

    table = comparison.compare(X, time, event,
                               feature_names=names,
                               test_size=0.2)
    table.insert(0, 'feature_set', label)
    return table, comparison

print('Running 12-month comparisons (3 feature sets × 4 models)...')

tbl_12m_all,  cmp_12m_all  = run_comparison(df_12m, feat_12m_all,  time, event, 365, 'All (rad+clin)')
tbl_12m_rad,  cmp_12m_rad  = run_comparison(df_12m, feat_12m_rad,  time, event, 365, 'Radiomics only')
tbl_12m_clin, cmp_12m_clin = run_comparison(df_12m, feat_12m_clin, time, event, 365, 'Clinical only')

results_12m = pd.concat([tbl_12m_all, tbl_12m_rad, tbl_12m_clin], ignore_index=True)
RESULTS['12m'] = results_12m

out = results_12m.copy()
if 'cv_auroc_mean' in out.columns and 'cv_auroc_std' in out.columns:
    out['cv_auroc'] = out.apply(
        lambda r: f"{r['cv_auroc_mean']:.3f} ± {r['cv_auroc_std']:.3f}", axis=1
    )
display_cols = ['feature_set', 'model', 'auroc', 'cv_auroc', 'brier_score']
available = [c for c in display_cols if c in out.columns]
print(out[available].to_string(index=False))

In [ ]:
# ROC curves — out-of-fold logistic predictions (n≈410, smooth curves for Figure 9)
# OOF predictions use 5-fold CV across the full cohort; smoother than n=82 test set.
# LASSO-selected features are already in each feature column set.

from sklearn.linear_model import LogisticRegression as LR

def get_oof_predictions(df, feat_cols, time_arr, event_arr, landmark_days,
                        n_folds=5, seed=42):
    """Return (y_true, y_prob_oof) using 5-fold out-of-fold logistic predictions."""
    cens_before = (event_arr == 0) & (time_arr < landmark_days)
    y = np.where(time_arr >= landmark_days, 1, 0)
    mask = ~cens_before
    X = df[feat_cols].values[mask]
    y = y[mask]

    oof_prob = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for train_idx, val_idx in skf.split(X, y):
        scaler = StandardScaler()
        X_tr  = scaler.fit_transform(X[train_idx])
        X_val = scaler.transform(X[val_idx])
        clf = LR(C=1.0, max_iter=1000, solver='lbfgs', random_state=seed)
        clf.fit(X_tr, y[train_idx])
        oof_prob[val_idx] = clf.predict_proba(X_val)[:, 1]

    return y, oof_prob

fig, ax = plt.subplots(figsize=(7, 6))

for feat_cols, label, color in [
    (feat_12m_rad,  'Radiomics only',       'darkorange'),
    (feat_12m_all,  'Radiomics + Clinical', 'steelblue'),
    (feat_12m_clin, 'Clinical baseline',    'green'),
]:
    y_oof, prob_oof = get_oof_predictions(df_12m, feat_cols, time, event, 365)
    fpr, tpr, _ = roc_curve(y_oof, prob_oof)
    auc = roc_auc_score(y_oof, prob_oof)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Chance')
ax.set_xlabel('1 − Specificity')
ax.set_ylabel('Sensitivity')
ax.set_title('12-Month OS Landmark — ROC Curves by Feature Set')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(HERE / 'roc_curves_12m.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_curves_12m.png')
print('Note: OOF AUCs above are from full-cohort 5-fold CV — reference values are from held-out test set in Table 4.2')

## 4. Model Comparison — 6-Month OS-as-PFS Proxy (Secondary)

> Class imbalance (84:16) — `class_weight='balanced'` applied within classifiers.

In [ ]:
print('Running 6-month comparisons...')

tbl_6m_all,  cmp_6m_all  = run_comparison(df_6m, feat_6m_all,  time, event, 180, 'All (rad+clin)')
tbl_6m_rad,  cmp_6m_rad  = run_comparison(df_6m, feat_6m_rad,  time, event, 180, 'Radiomics only')
tbl_6m_clin, cmp_6m_clin = run_comparison(df_6m, feat_6m_clin, time, event, 180, 'Clinical only')

results_6m = pd.concat([tbl_6m_all, tbl_6m_rad, tbl_6m_clin], ignore_index=True)
RESULTS['6m'] = results_6m

out_6m = results_6m.copy()
if 'cv_auroc_mean' in out_6m.columns and 'cv_auroc_std' in out_6m.columns:
    out_6m['cv_auroc'] = out_6m.apply(
        lambda r: f"{r['cv_auroc_mean']:.3f} ± {r['cv_auroc_std']:.3f}", axis=1
    )
display_cols_6m = ['feature_set', 'model', 'auroc', 'cv_auroc', 'brier_score']
available_6m = [c for c in display_cols_6m if c in out_6m.columns]
print(out_6m[available_6m].to_string(index=False))

## 5. Combined Results Table

In [ ]:
results_12m['landmark'] = '12-month OS'
results_6m['landmark']  = '6-month OS-as-PFS'

combined = pd.concat([results_12m, results_6m], ignore_index=True)
combined.to_csv(HERE / 'classifier_results.csv', index=False)

# Pivot: AUROC by model and feature set
if 'auroc' in combined.columns:
    pivot = combined.pivot_table(
        index=['landmark','feature_set'],
        columns='model',
        values='auroc',
        aggfunc='mean'
    ).round(3)
    print('AUROC summary:')
    print(pivot.to_string())

print('\nSaved: classifier_results.csv')

## 6. Stage-Stratified Analysis

Model performance by TNM stage group — does radiomic signal add value beyond stage alone?

In [ ]:
# Load master cohort for stage info
master = pd.read_csv(HERE / 'master_cohort.csv',
                     usecols=['patient_id','overall_stage'])
stage_map = {'I':1,'II':2,'IIIa':3,'IIIb':4}
master['stage_n'] = master['overall_stage'].map(stage_map)

df_12m_s = df_12m.merge(master[['patient_id','overall_stage','stage_n']],
                        on='patient_id', how='left')

stage_groups = {'Early (I/II)': [1,2], 'Advanced (IIIa/IIIb)': [3,4]}

stage_results = []
for grp_name, stage_vals in stage_groups.items():
    mask = df_12m_s['stage_n'].isin(stage_vals).values
    n = mask.sum()
    if n < 20:
        print(f'{grp_name}: too few patients ({n}), skipping')
        continue

    X_sub   = df_12m_s.loc[mask, feat_12m_all].values
    time_sub  = time[mask]
    event_sub = event[mask]

    try:
        config = ClassifierConfig(
            model_type=ModelType.XGBOOST,
            landmark_days=365,
            censoring_strategy=CensoringStrategy.EXCLUDE,
            n_folds=3,
            random_state=42,
        )
        pred = ResponsePredictor(config)
        result = pred.fit_evaluate(X_sub, time_sub, event_sub,
                                   feature_names=feat_12m_all,
                                   test_size=0.2)
        auroc = result.metrics.get('auroc', float('nan'))
        stage_results.append({'Stage group': grp_name, 'N': n, 'AUROC': round(auroc, 3)})
    except Exception as e:
        stage_results.append({'Stage group': grp_name, 'N': n, 'AUROC': f'Error: {e}'})

print('Stage-stratified AUROC (XGBoost, 12-month, all features):')
print(pd.DataFrame(stage_results).to_string(index=False))

## 7. Decision Curve Analysis

Net benefit of the best model versus clinical-only baseline and treat-all/treat-none strategies.
Answers RQ3: does the model provide positive net benefit over the clinical baseline?

In [ ]:
def decision_curve(y_true, y_prob_model, y_prob_clinical,
                   thresholds=None, ax=None, title=''):
    """Simple decision curve analysis."""
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 50)

    n = len(y_true)
    prev = y_true.mean()

    def net_benefit(probs, thresh):
        predicted_pos = probs >= thresh
        tp = (predicted_pos & y_true.astype(bool)).sum()
        fp = (predicted_pos & ~y_true.astype(bool)).sum()
        return (tp / n) - (fp / n) * (thresh / (1 - thresh))

    nb_model    = [net_benefit(y_prob_model,    t) for t in thresholds]
    nb_clinical = [net_benefit(y_prob_clinical, t) for t in thresholds]
    nb_all      = [prev - (1 - prev) * t / (1 - t) for t in thresholds]
    nb_none     = [0] * len(thresholds)

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(thresholds, nb_model,    label='Radiomics-only logistic', color='steelblue', lw=2)
    ax.plot(thresholds, nb_clinical, label='Clinical baseline',        color='green',     lw=2, linestyle='--')
    ax.plot(thresholds, nb_all,      label='Treat all',                color='grey',      lw=1, linestyle=':')
    ax.plot(thresholds, nb_none,     label='Treat none',               color='black',     lw=1, linestyle='-.')
    ax.set_xlim(0.05, 0.95)
    ax.set_ylim(-0.05, None)
    ax.set_xlabel('Decision threshold')
    ax.set_ylabel('Net benefit')
    ax.set_title(title)
    ax.legend(fontsize=9)

    # Print threshold range where model exceeds clinical baseline
    thresholds_arr = np.array(thresholds)
    nb_model_arr   = np.array(nb_model)
    nb_clin_arr    = np.array(nb_clinical)
    positive_range = thresholds_arr[(nb_model_arr > nb_clin_arr) & (nb_model_arr > 0)]
    if len(positive_range):
        print(f'  {title}: model exceeds clinical baseline '
              f'pt={positive_range[0]:.2f} to pt={positive_range[-1]:.2f}')
    else:
        print(f'  {title}: model does not exceed clinical baseline')

    return ax

# DCA uses logistic regression (radiomics-only) — consistent with primary analysis model
print('Fitting models for DCA (logistic, radiomics-only vs clinical logistic)...')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (df, feat_rad_grp, feat_clin, lm, title) in enumerate([
    (df_12m, feat_12m_rad, feat_12m_clin, 365, '12-month OS (primary)'),
    (df_6m,  feat_6m_rad,  feat_6m_clin,  180, '6-month OS-as-PFS (secondary)'),
]):
    config = ClassifierConfig(
        model_type=ModelType.LOGISTIC,
        landmark_days=lm,
        censoring_strategy=CensoringStrategy.EXCLUDE,
        n_folds=5, random_state=42
    )

    try:
        pred_rad  = ResponsePredictor(config)
        res_rad   = pred_rad.fit_evaluate(df[feat_rad_grp].values, time, event,
                                          feature_names=feat_rad_grp, test_size=0.2)

        pred_clin = ResponsePredictor(config)
        res_clin  = pred_clin.fit_evaluate(df[feat_clin].values, time, event,
                                           feature_names=feat_clin, test_size=0.2)

        if res_rad.y_prob is not None and res_clin.y_prob is not None:
            decision_curve(
                y_true=res_rad.y_true,
                y_prob_model=res_rad.y_prob,
                y_prob_clinical=res_clin.y_prob,
                ax=axes[i], title=title
            )
    except Exception as e:
        axes[i].text(0.5, 0.5, f'DCA error:\n{e}',
                     transform=axes[i].transAxes, ha='center')

plt.suptitle('Decision Curve Analysis', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(HERE / 'dca_plot_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dca_plot_v2.png')

## 8. Save Results

In [ ]:
combined.to_csv(HERE / 'classifier_results.csv', index=False)

# Summary for thesis
summary = {
    'cohort_n': len(df_12m),
    'landmark_12m': {
        'features_all': len(feat_12m_all),
        'features_rad': len(feat_12m_rad),
        'features_clin': len(feat_12m_clin),
    },
    'landmark_6m': {
        'features_all': len(feat_6m_all),
        'features_rad': len(feat_6m_rad),
        'features_clin': len(feat_6m_clin),
    },
}

with open(HERE / 'classifier_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved: classifier_results.csv')
print('Saved: classifier_summary.json')
print('\nNext: update Results chapter in 04_Results.md')